# Lab AWS — Titanic: SageMaker Processing Job

## 1. Preparación del entorno

In [ ]:
import boto3
import sagemaker
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

sess   = sagemaker.Session()
bucket = sess.default_bucket()
prefix = "sagemaker/titanic"
role   = boto3.client('iam').get_role(RoleName='LabRole')['Role']['Arn']

sklearn_image_uri = sagemaker.image_uris.retrieve(
    framework="sklearn",
    region=sess.boto_region_name,
    version="1.2-1",
    py_version="py3",
    instance_type="ml.m5.xlarge",
)

print(f"Bucket    : s3://{bucket}")
print(f"Region    : {sess.boto_region_name}")
print(f"Role      : {role}")
print(f"Imagen    : {sklearn_image_uri}")

## 2. Descargar dataset Titanic y subirlo a S3

In [ ]:
import urllib.request
import os

RAW_URL  = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
RAW_FILE = "titanic.csv"

urllib.request.urlretrieve(RAW_URL, RAW_FILE)
print(f"Descargado: {os.path.getsize(RAW_FILE)} bytes")

s3_raw_uri = sess.upload_data(
    path=RAW_FILE,
    bucket=bucket,
    key_prefix=f"{prefix}/raw"
)
print(f"En S3: {s3_raw_uri}")

## 3. Definir el ScriptProcessor con imagen sklearn

In [ ]:
processor = ScriptProcessor(
    image_uri=sklearn_image_uri,
    command=["python3"],
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    base_job_name="titanic-processing",
    sagemaker_session=sess,
)

print("Processor configurado.")

## 4. Lanzar el Processing Job

Tarda entre **5 y 10 minutos**. El `[*]` indica que está corriendo.

In [ ]:
processor.run(
    code="processing.py",
    inputs=[
        ProcessingInput(
            source=s3_raw_uri,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            source="/opt/ml/processing/output/train",
            destination=f"s3://{bucket}/{prefix}/train",
            output_name="train"
        ),
        ProcessingOutput(
            source="/opt/ml/processing/output/validation",
            destination=f"s3://{bucket}/{prefix}/validation",
            output_name="validation"
        ),
        ProcessingOutput(
            source="/opt/ml/processing/output/test",
            destination=f"s3://{bucket}/{prefix}/test",
            output_name="test"
        ),
    ],
)

print("Processing job finalizado.")

## 5. Verificar outputs en S3

In [ ]:
s3 = boto3.client("s3")

for split in ["train", "validation", "test"]:
    key = f"{prefix}/{split}/{split}.csv"
    obj = s3.head_object(Bucket=bucket, Key=key)
    print(f"s3://{bucket}/{key}  ->  {obj['ContentLength']/1024:.1f} KB")

print(f"\ntrain      : s3://{bucket}/{prefix}/train")
print(f"validation : s3://{bucket}/{prefix}/validation")